# Kova — OmniVoice Studio worker (Google Colab GPU)

Notebook này chỉ chạy worker lồng tiếng trên **Google Colab GPU**. Kova Desktop trên Windows chỉ gửi yêu cầu HTTP qua URL tunnel; notebook không chạy inference, TTS hay clone giọng trên máy Windows.

## Cách dùng

1. Trong Colab chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU NVIDIA khác).
2. Chọn **Runtime → Run all**. Cell đầu tiên sẽ dừng ngay nếu không có CUDA.
3. Chờ cell cuối in `https://…trycloudflare.com` và **Session token**. Dán cả hai vào **Kova Desktop → Cấu hình AI → OmniVoice Worker URL / Session token**, rồi bấm kiểm tra kết nối. Token chỉ hợp lệ trong runtime Colab hiện tại và không được lưu vào config.
4. Giữ tab Colab mở trong lúc Kova tạo audio. URL sẽ hết hạn nếu runtime/tunnel ngắt.

Notebook clone repository công khai `https://github.com/debpalash/OmniVoice-Studio`; không phụ thuộc vào một Kova repository chưa được publish công khai. Voice profile chỉ được tạo trên Colab khi Desktop đã chọn audio tham chiếu và xác nhận quyền sử dụng giọng.


In [ ]:
# Cell 1 — GPU gate. CPU is intentionally not supported.
import shutil
import subprocess

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Kova requires a Colab NVIDIA GPU. Select Runtime → Change runtime type → T4 GPU, then Run all again.')

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('CUDA is not available in this Colab runtime. Select a GPU runtime and Run all again.')

print(gpu.stdout)
print('✅ NVIDIA GPU detected. Continuing with GPU-only setup.')


In [ ]:
# Cell 2 — Clone the public OmniVoice Studio repository and install the API worker.
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/debpalash/OmniVoice-Studio.git'
REPO_DIR = '/content/OmniVoice-Studio'
os.environ.setdefault('UV_HTTP_TIMEOUT', '300')

def run(cmd, *, cwd=None, what='command'):
    shown = cmd if isinstance(cmd, str) else ' '.join(cmd)
    print(f'\n$ {shown}')
    completed = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str))
    if completed.returncode != 0:
        raise RuntimeError(f'{what} failed with exit code {completed.returncode}: {shown}')

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, what='update OmniVoice Studio')
else:
    run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], what='clone OmniVoice Studio')

run('apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1', what='install system audio packages')
if not shutil.which('uv'):
    run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], what='install uv')
run(['uv', 'pip', 'install', '--system', '--no-cache', '.'], cwd=REPO_DIR, what='install OmniVoice Studio')
run([sys.executable, os.path.join(REPO_DIR, 'scripts', 'setup.py')], what='configure CUDA compatibility')
run([sys.executable, '-c', 'import torch; assert torch.cuda.is_available(), \"CUDA is unavailable after install\"; print(\"✅ PyTorch CUDA: \", torch.cuda.get_device_name(0))'], what='verify PyTorch CUDA')


In [ ]:
# Cell 3 — Start OmniVoice Studio on the Colab GPU.
import json
import os
import secrets
import subprocess
import sys
import time
import urllib.request

STUDIO_PORT = 3900
STUDIO_HEALTH = f'http://127.0.0.1:{STUDIO_PORT}/health'
STUDIO_LOG = '/content/kova_omnivoice_studio.log'
# Generated once per runtime. Do not put a chosen token in this notebook.
KOVA_SESSION_TOKEN = os.environ.get('KOVA_SESSION_TOKEN') or secrets.token_urlsafe(32)
os.environ['KOVA_SESSION_TOKEN'] = KOVA_SESSION_TOKEN
STUDIO_HEADERS = {'Authorization': 'Bearer ' + KOVA_SESSION_TOKEN}

def get_json(url, timeout=5, headers=None):
    request = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)

try:
    studio_health = get_json(STUDIO_HEALTH, headers=STUDIO_HEADERS)
except Exception:
    studio_health = None

if not studio_health:
    subprocess.run(f'kill -9 $(lsof -t -i:{STUDIO_PORT}) 2>/dev/null || true', shell=True)
    env = os.environ.copy()
    env.update({
        'OMNIVOICE_SERVER_MODE': '1',
        'OMNIVOICE_API_KEY': KOVA_SESSION_TOKEN,
        # Never silently fall back to CPU if this Colab GPU is incompatible.
        'OMNIVOICE_FORCE_CUDA': '1',
        # The first real TTS synthesis may download/load weights on Colab.
        'OMNIVOICE_SELFTEST_TIMEOUT_S': '600',
        'OMNIVOICE_DATA_DIR': '/content/kova_omnivoice_data',
        'PYTHONUNBUFFERED': '1',
    })
    log = open(STUDIO_LOG, 'ab')
    studio_process = subprocess.Popen([
        sys.executable, '-m', 'uvicorn', 'main:app',
        '--app-dir', 'backend', '--host', '127.0.0.1', '--port', str(STUDIO_PORT),
    ], cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline:
        if studio_process.poll() is not None:
            break
        try:
            studio_health = get_json(STUDIO_HEALTH, headers=STUDIO_HEADERS)
            break
        except Exception:
            time.sleep(3)

if not studio_health:
    tail = open(STUDIO_LOG, 'r', errors='replace').read()[-5000:] if os.path.exists(STUDIO_LOG) else '(no log)'
    raise RuntimeError(f'OmniVoice Studio did not start.\n--- log tail ---\n{tail}')
if 'cuda' not in str(studio_health.get('device', '')).lower():
    raise RuntimeError(f'OmniVoice Studio is not on CUDA: {studio_health}. Stop here; do not use CPU fallback.')
print('✅ OmniVoice Studio GPU worker is ready:', studio_health)


In [ ]:
# Cell 4 — Start a small Kova compatibility adapter (only inside Colab).
# It gives Kova a strict {ready, device} health contract and proxies profile/generate API calls to Studio.
from pathlib import Path

ADAPTER_FILE = Path('/content/kova_omnivoice_adapter.py')
ADAPTER_FILE.write_text(r'''
import hmac
import os
import httpx
from pathlib import Path
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response

UPSTREAM = 'http://127.0.0.1:3900'
WORKER_TOKEN = os.environ['KOVA_SESSION_TOKEN']
EXPECTED_AUTHORIZATION = 'Bearer ' + WORKER_TOKEN
PREFLIGHT_READY_FILE = Path('/content/kova_omnivoice_preflight.ready')
app = FastAPI(title='Kova Colab OmniVoice Adapter')

def reject_if_unauthorized(request: Request):
    supplied = request.headers.get('authorization', '')
    if not hmac.compare_digest(supplied, EXPECTED_AUTHORIZATION):
        return JSONResponse({'detail': 'missing or invalid worker session token'}, status_code=401)
    return None

@app.get('/health')
async def health(request: Request):
    rejected = reject_if_unauthorized(request)
    if rejected:
        return rejected
    try:
        async with httpx.AsyncClient(timeout=15) as client:
            response = await client.get(UPSTREAM + '/health', headers={'Authorization': EXPECTED_AUTHORIZATION})
            response.raise_for_status()
            upstream = response.json()
    except Exception as exc:
        return JSONResponse({'status': 'error', 'ready': False, 'detail': str(exc)}, status_code=503)
    device = str(upstream.get('device', ''))
    if upstream.get('status') != 'ok' or 'cuda' not in device.lower():
        return JSONResponse({'status': 'error', 'ready': False, 'device': device}, status_code=503)
    if not PREFLIGHT_READY_FILE.is_file():
        return {'status': 'warming', 'ready': False, 'device': device, 'dtype': 'cuda'}
    return {'status': 'ready', 'ready': True, 'device': device, 'dtype': 'cuda'}

@app.api_route('/{path:path}', methods=['GET', 'POST', 'PUT', 'PATCH', 'DELETE', 'OPTIONS'])
async def proxy(path: str, request: Request):
    rejected = reject_if_unauthorized(request)
    if rejected:
        return rejected
    body = await request.body()
    request_headers = {key: value for key, value in request.headers.items() if key.lower() not in {'host', 'content-length', 'connection'}}
    request_headers['authorization'] = EXPECTED_AUTHORIZATION
    async with httpx.AsyncClient(timeout=1800) as client:
        upstream = await client.request(request.method, f'{UPSTREAM}/{path}', content=body, headers=request_headers)
    response_headers = {key: value for key, value in upstream.headers.items() if key.lower() not in {'content-length', 'transfer-encoding', 'connection'}}
    return Response(content=upstream.content, status_code=upstream.status_code, headers=response_headers)
'''.lstrip(), encoding='utf-8')

ADAPTER_PORT = 11435
ADAPTER_HEALTH = f'http://127.0.0.1:{ADAPTER_PORT}/health'
ADAPTER_LOG = '/content/kova_omnivoice_adapter.log'
ADAPTER_HEADERS = {'Authorization': 'Bearer ' + KOVA_SESSION_TOKEN}
PREFLIGHT_READY_FILE = Path('/content/kova_omnivoice_preflight.ready')
PREFLIGHT_READY_FILE.unlink(missing_ok=True)
# Always restart the tiny adapter: the token and generated source are per Run all.
subprocess.run(f'kill -9 $(lsof -t -i:{ADAPTER_PORT}) 2>/dev/null || true', shell=True)
adapter_log = open(ADAPTER_LOG, 'ab')
adapter_process = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'kova_omnivoice_adapter:app',
    '--app-dir', '/content', '--host', '127.0.0.1', '--port', str(ADAPTER_PORT),
], stdout=adapter_log, stderr=subprocess.STDOUT)
adapter_health = None
deadline = time.time() + 60
while time.time() < deadline:
    try:
        adapter_health = get_json(ADAPTER_HEALTH, headers=ADAPTER_HEADERS)
        break
    except Exception:
        time.sleep(2)

if not adapter_health or 'cuda' not in str(adapter_health.get('device', '')).lower():
    raise RuntimeError(f'Kova adapter did not start on CUDA: {adapter_health}')

def post_json(url, timeout=660, headers=None):
    request_headers = dict(headers or {})
    request_headers['Content-Type'] = 'application/json'
    request = urllib.request.Request(url, data=b'', headers=request_headers, method='POST')
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)

# This authenticated request performs real, short design-voice synthesis.
# It uses no user reference audio/profile and gates the public health route.
preflight = post_json(f'http://127.0.0.1:{ADAPTER_PORT}/engines/omnivoice/selftest', headers=ADAPTER_HEADERS)
if not preflight.get('ok') or int(preflight.get('num_samples') or 0) <= 0:
    raise RuntimeError(f'OmniVoice GPU synthesis preflight failed: {preflight}')
PREFLIGHT_READY_FILE.write_text(json.dumps({'engine': preflight.get('id'), 'samples': preflight.get('num_samples')}), encoding='utf-8')
adapter_health = get_json(ADAPTER_HEALTH, headers=ADAPTER_HEADERS)
if not adapter_health.get('ready') or 'cuda' not in str(adapter_health.get('device', '')).lower():
    raise RuntimeError(f'Kova adapter is not CUDA-ready after synthesis preflight: {adapter_health}')
print('✅ Kova adapter completed real GPU TTS preflight:', preflight)


In [ ]:
# Cell 5 — Create a public HTTPS tunnel and print the exact URL to paste into Kova Desktop.
import re

CLOUDFLARED = '/content/cloudflared'
TUNNEL_LOG = '/content/kova_cloudflared.log'
if not os.path.isfile(CLOUDFLARED):
    run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', CLOUDFLARED], what='download cloudflared')
    run(['chmod', '+x', CLOUDFLARED], what='make cloudflared executable')

subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
with open(TUNNEL_LOG, 'w') as log:
    tunnel_process = subprocess.Popen([
        CLOUDFLARED, 'tunnel', '--url', f'http://127.0.0.1:{ADAPTER_PORT}', '--no-autoupdate'
    ], stdout=log, stderr=subprocess.STDOUT)

tunnel_url = None
deadline = time.time() + 90
while time.time() < deadline:
    time.sleep(2)
    text = open(TUNNEL_LOG, 'r', errors='replace').read()
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
    if matches:
        tunnel_url = matches[-1]
        break

if not tunnel_url:
    raise RuntimeError('Cloudflare tunnel URL was not produced within 90 seconds.\n' + open(TUNNEL_LOG, 'r', errors='replace').read()[-4000:])

public_health = get_json(tunnel_url + '/health', timeout=30, headers={'Authorization': 'Bearer ' + KOVA_SESSION_TOKEN})
if not public_health.get('ready') or 'cuda' not in str(public_health.get('device', '')).lower():
    raise RuntimeError(f'Public tunnel did not expose a CUDA-ready Kova worker: {public_health}')

print('\n' + '=' * 72)
print('KOVA COLAB GPU WORKER IS READY')
print('PASTE THIS HTTPS URL INTO KOVA DESKTOP:')
print(tunnel_url)
print('PASTE THIS TEMPORARY SESSION TOKEN INTO KOVA DESKTOP (it is not saved):')
print(KOVA_SESSION_TOKEN)
print('=' * 72)
print('Keep this Colab tab open. Kova will upload the selected reference once, only after explicit consent, and reuse its remote profile for the whole job.')
